## Writing Viterbi Algorithm for the Primer
### Key Components

1. **Precision Setup**:
  - Uses Python's `Decimal` module for high-precision arithmetic (50 decimal places)
  - This prevents underflow issues when multiplying many small probabilities

2. **HMM Model Definition**:
  - **States**: 'Start' (initial), 'E' (exon), '5' (splice site), 'I' (intron)
  - **Transition Probabilities**: Defines state-to-state transition probabilities
    - Example: Exon (E) has 0.9 probability of staying in E, 0.1 of transitioning to splice site (5)
  - **Emission Probabilities**: Defines probability of emitting each nucleotide from each state
    - Example: Splice site (5) state emits 'G' with 0.95 probability

3. **Probability Calculation Function**:
  - Takes a state path and observed sequence as inputs
  - Verifies that state path and sequence have matching lengths
  - Initializes with probability 1.0 and 'Start' as first state
  - For each position in the sequence:
    - Multiplies by transition probability from previous state to current state
    - Multiplies by emission probability of the observed nucleotide from current state
  - Special handling for ending in state 'I' (multiplies by 0.1 termination probability)
  - Converts final probability to logarithmic form using natural logarithm

4. **Sample Application**:
  - Tests the function with a specific state path "EEEEEEEEEEEEEEEEEE5IIIIIII"
  - And corresponding nucleotide sequence "CTTCATGTGAAAGCAGACGTAAGTCA"
  - Outputs the logarithmic probability (rounded to 2 decimal places)

In [1]:
from decimal import Decimal, getcontext
import math

getcontext().prec = 50

def get_log_prob_of_a_given_path(state_path, sequence):
    assert len(state_path) == len(sequence), "State path and sequence must be of the same length."
    
    # Define transitions
    transitions = {
        'Start': {'E': Decimal('1.0')},
        'E': {'E': Decimal('0.9'), '5': Decimal('0.1')},
        '5': {'I': Decimal('1.0')},
        'I': {'I': Decimal('0.9')}
    }

    # Define emissions 
    emissions = {
        'E': {'A': Decimal('0.25'), 'C': Decimal('0.25'), 'G': Decimal('0.25'), 'T': Decimal('0.25')},
        '5': {'A': Decimal('0.05'), 'C': Decimal('0.0'), 'G': Decimal('0.95'), 'T': Decimal('0.0')},
        'I': {'A': Decimal('0.4'), 'C': Decimal('0.1'), 'G': Decimal('0.1'), 'T': Decimal('0.4')}
    }

    prob = Decimal('1.0')
    prev_state = 'Start'
    
    for i in range(len(sequence)):
        state = state_path[i]
        symbol = sequence[i]
        prob *= transitions[prev_state][state]
        prob *= emissions[state][symbol]
        prev_state = state
    if prev_state == 'I':
        prob*=Decimal('0.1')
    # Compute the natural log 
    log_prob = float(prob.ln())
    return prob, log_prob

state_path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA" 

prod, log_prob = get_log_prob_of_a_given_path(state_path, sequence)
print("Log probability =", round(log_prob,2))



Log probability = -41.22


## Using Viterbi Algorithm to find maximum likely path
### Model Definition

1. **States**: Three biological states are defined:
  - 'E': Exon regions (coding DNA)
  - '5': 5' splice site (transition point from exon to intron)
  - 'I': Intron regions (non-coding DNA)

2. **Transition Probabilities**:
  - Starting state: Always begins with an exon (probability 1.0)
  - Exon transitions: Can stay in exon state (0.9) or transition to splice site (0.1)
  - Splice site: Always transitions to intron (probability 1.0)
  - Intron: Can stay in intron state (0.9) or terminate (implied 0.1)

3. **Emission Probabilities**:
  - Exon: Equal probability (0.25) of emitting any nucleotide
  - Splice site: Strong preference for 'G' (0.95), small chance of 'A' (0.05)
  - Intron: Preference for 'A' and 'T' (0.4 each), lower probability for 'C' and 'G' (0.1 each)

### Algorithm Implementation

1. **Initialization**:
  - Sets up data structures to track probabilities and paths
  - Initializes first position with only exon state possible
  - Uses logarithmic probabilities to prevent underflow

2. **Dynamic Programming Recursion**:
  - For each position in the sequence:
    - For each possible current state:
      - Evaluates all possible previous states
      - Computes probability of transitioning from previous state and emitting current nucleotide
      - Selects the highest probability path
      - Stores both the probability and the state path

3. **Termination Handling**:
  - For intron end states, applies additional 0.1 termination probability
  - Identifies the highest probability final state

4. **Output**:
  - Returns the most likely state path as a string (e.g., "EEEEEE5IIIII")
  - Returns the logarithmic probability of this path

### Mathematical Foundation

The Viterbi algorithm recursively computes:
- V[t][state] = max(V[t-1][prev_state] + log(transition_prob) + log(emission_prob))

This represents the maximum log probability of the best path ending in 'state' at position 't'.


In [5]:

getcontext().prec = 50

def viterbi(sequence):
    states = ['E', '5', 'I']
    
    transitions = {
        'Start': {'E': Decimal('1.0')},
        'E': {'E': Decimal('0.9'), '5': Decimal('0.1')},
        '5': {'I': Decimal('1.0')},
        'I': {'I': Decimal('0.9')}
    }
    
    emissions = {
        'E': {'A': Decimal('0.25'), 'C': Decimal('0.25'), 'G': Decimal('0.25'), 'T': Decimal('0.25')},
        '5': {'A': Decimal('0.05'), 'C': Decimal('0.0'), 'G': Decimal('0.95'), 'T': Decimal('0.0')},
        'I': {'A': Decimal('0.4'), 'C': Decimal('0.1'), 'G': Decimal('0.1'), 'T': Decimal('0.4')}
    }
    
    n = len(sequence)
    # V[t][state]: best log probability for a path ending in state at position t.
    V = [{} for _ in range(n)]
    # path[state] will record the best path (list of states) to that state at time t
    path = {}
    initial = 'E'
    first_obs = sequence[0]
    emiss_prob = emissions[initial].get(first_obs, Decimal('0'))
    if emiss_prob == 0:
        V[0][initial] = Decimal('-Infinity')
    else:
        V[0][initial] = transitions['Start'][initial].ln() + emiss_prob.ln()
    path[initial] = [initial]
    
    for s in states:
        if s != initial:
            V[0][s] = Decimal('-Infinity')
            path[s] = []

    for t in range(1, n):
        new_V = {}
        new_path = {}
        obs = sequence[t]
        for curr in states:
            best_prob = Decimal('-Infinity')
            best_prev = None
            for prev in states:
                if curr not in transitions.get(prev, {}):
                    continue
                trans_prob = transitions[prev][curr]
                emiss_prob = emissions[curr].get(obs, Decimal('0'))
                if emiss_prob == 0:
                    continue
                candidate = V[t-1][prev] + trans_prob.ln() + emiss_prob.ln()
                if candidate > best_prob:
                    best_prob = candidate
                    best_prev = prev
            new_V[curr] = best_prob
            if best_prev is not None and path[best_prev]:
                new_path[curr] = path[best_prev] + [curr]
            else:
                new_path[curr] = []
        V[t] = new_V
        path = new_path
    
    final_best_state = None
    final_log_prob = Decimal('-Infinity')
    for s in states:
        if s == 'I':
            candidate = V[n-1][s] + Decimal('0.1').ln()
        else:
            candidate = V[n-1][s]

        if candidate > final_log_prob:
            final_log_prob = candidate
            final_best_state = s
    best_path = path[final_best_state]
    
    return ''.join(best_path), float(final_log_prob)

# Primers Example:
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA" 
best_state_path, log_probability = viterbi(sequence)
print("Best state path:", best_state_path)
print("Log probability:",round(log_probability,2))


Best state path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability: -38.68
